# 05 · Minería de datos (KDD / CRISP-DM)

| Fase CRISP-DM | Dónde |
|---|---|
| Comprensión del negocio | ETA fiable y detección de zonas críticas para redistribuir rutas |
| Comprensión y preparación de datos | ETL al DW (`04_olap`) · `mineria.cargar_*` |
| Modelado | Regresión lineal (predictivo) · DBSCAN y K-Means (descriptivo) |
| Evaluación | MAE / R² contra línea base · contraste con zonas conocidas |
| Despliegue | Centroides como candidatos a centros de transferencia (*edge*) |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from logistica import etl, mineria
from logistica.geo import haversine_km
from logistica.semilla import ZONAS_CRITICAS

pd.set_option("display.width", 160)
etl.ejecutar_etl()

## A. Modelado predictivo — ETA con regresión lineal

**Por qué regresión lineal.** El tiempo de entrega es continuo y aproximadamente lineal en distancia,
tráfico y lluvia; el modelo es interpretable (cada coeficiente son horas por unidad) y es la línea
base natural antes de pasar a una red recurrente (LSTM) cuando haya series temporales de GPS densas.

In [ ]:
entregas = mineria.cargar_entregas()
print(len(entregas), "entregas")
display(entregas.describe().round(2))

In [ ]:
eta = mineria.entrenar_eta(entregas)
display(
    pd.Series(
        {
            "MAE modelo (h)": eta.mae_horas,
            "MAE línea base: predecir la media (h)": eta.mae_base_horas,
            "R²": eta.r2,
        }
    )
    .round(3)
    .to_frame("test 20 %")
)
display(eta.coeficientes.round(3).to_frame("coeficiente (horas)").head(8))

In [ ]:
x = entregas.drop(columns=["envio_id", "horas_entrega", "anio"])
real, predicho = entregas.horas_entrega, eta.modelo.predict(x)
fig, ax = plt.subplots(figsize=(5, 4.5))
ax.scatter(real, predicho, s=6, alpha=0.4)
tope = max(real.max(), predicho.max())
ax.plot([0, tope], [0, tope], "k--", lw=1)
ax.set_xlabel("horas reales")
ax.set_ylabel("horas predichas (ETA)")
plt.tight_layout()
plt.show()

## B. Modelado descriptivo — puntos calientes de incidentes

**DBSCAN** (densidad, distancia haversine): no exige fijar el número de zonas y deja los incidentes
aislados como ruido (`-1`). **K-Means** sobre las mismas coordenadas propone ubicaciones concretas para
nuevos centros de transferencia; el método del codo elige `k`.

In [ ]:
incidentes = mineria.cargar_incidentes()
etiquetados = mineria.detectar_puntos_calientes(incidentes, radio_km=25, minimo_incidentes=15)
resumen = mineria.resumir_clusters(etiquetados)
print(
    f"{len(incidentes)} incidentes · {len(resumen)} zonas críticas · {(etiquetados.cluster == -1).sum()} aislados (ruido)"
)
display(resumen)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 6.5))
ruido = etiquetados[etiquetados.cluster == -1]
ax.scatter(ruido.lon, ruido.lat, s=6, c="lightgray", label="ruido")
for c, grupo in etiquetados[etiquetados.cluster >= 0].groupby("cluster"):
    ax.scatter(grupo.lon, grupo.lat, s=10, label=f"zona {c} ({len(grupo)})")
ax.legend(fontsize=7)
ax.set_xlabel("longitud")
ax.set_ylabel("latitud")
plt.tight_layout()
plt.show()

In [ ]:
codo = mineria.inercia_por_k(
    incidentes[incidentes.index.isin(etiquetados[etiquetados.cluster >= 0].index)]
)
ax = codo.plot(marker="o", figsize=(5, 3))
ax.set_xlabel("k")
ax.set_ylabel("inercia")
plt.tight_layout()
plt.show()

In [ ]:
n_zonas = sum(len(z) for z in ZONAS_CRITICAS.values())
centros = mineria.candidatos_centros_transferencia(etiquetados[etiquetados.cluster >= 0], k=n_zonas)
display(centros)

reales = [(lat, lon) for zonas in ZONAS_CRITICAS.values() for lat, lon, _, _ in zonas]
error = [min(haversine_km(c.lat, c.lon, la, lo) for la, lo in reales) for c in centros.itertuples()]
print("Distancia de cada centroide a la zona crítica sembrada (km):", np.round(error, 1))

**Evaluación.** Cada centroide cae a pocos kilómetros de una zona sembrada: el clustering recupera la
estructura que se inyectó. Con datos reales no hay "verdad conocida"; se validaría con el negocio
(¿coinciden con tramos de ruta problemáticos?) y con métricas internas como la silueta.